In [1]:
import requests
import os
from dotenv import load_dotenv
import pandas as pd
import plotly.express as px
import streamlit as st

load_dotenv()


False

In [2]:
#
api_key=os.getenv("api_key")


url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {"function":"TIME_SERIES_DAILY","symbol":"MSFT","outputsize":"compact","datatype":"json"}

headers = {
	"x-rapidapi-key": "29d04bb977msh6e6ccf157115028p116a36jsndf3231f7d1da",
	"x-rapidapi-host": "alpha-vantage.p.rapidapi.com",
	"Content-Type": "application/json"
}

response = requests.get(url, headers=headers, params=querystring)

if response.status_code==200:
    data=response.json()
else:
    print(response.status_code)

df=pd.DataFrame(data)
df

,Meta Data,Time Series (Daily)
1. Information,"Daily Prices (open, high, low, close) and Volumes",NaN
2. Symbol,MSFT,NaN
3. Last Refreshed,2026-10-27,NaN
4. Output Size,Compact,NaN
5. Time Zone,US/Eastern,NaN
...,...,...
2026-06-11,NaN,"{'1. open': '395.2050', '2. high': '396.8500',..."
2026-06-10,NaN,"{'1. open': '398.5500', '2. high': '405.0400',..."
2026-06-09,NaN,"{'1. open': '409.0300', '2. high': '411.9800',..."
2026-06-08,NaN,"{'1. open': '414.1400', '2. high': '417.1599',..."


In [3]:
df = pd.DataFrame.from_dict(data['Time Series (Daily)'], orient='index')

# 3. Clean the column names (removing "1. ", "2. ", etc.)
# Instead of '1. open', it becomes just 'open'
df.columns = [c.split('. ')[1] for c in df.columns]

# 4. Convert all columns to numeric (they come as strings from the API)
df = df.apply(pd.to_numeric)

# 5. Convert the index to actual Datetime objects
df.index = pd.to_datetime(df.index)
df

,open,high,low,close,volume
2026-10-27,532.350,534.7000,528.33,531.33,19532321
2026-10-26,527.400,533.2500,521.69,532.60,19021036
2026-10-23,517.745,525.7700,515.70,524.51,13681067
2026-10-22,533.140,533.5960,520.30,521.08,21266610
2026-10-21,529.440,533.2800,525.02,532.95,13940102
...,...,...,...,...,...
2026-06-11,395.205,396.8500,384.00,390.34,47224135
2026-06-10,398.550,405.0400,397.16,397.36,32576041
2026-06-09,409.030,411.9800,398.48,403.41,35317302
2026-06-08,414.140,417.1599,408.56,411.74,32086714


In [4]:
# Starting the EDA 
# 1. Inspecting the data structure
print("Here is the Structural info about the data : ")
print(df.info())
print("Inspecting the numerical summmary of the data :")
print(df.describe())

Here is the Structural info about the data : 
<class 'pandas.DataFrame'>
DatetimeIndex: 100 entries, 2026-10-27 to 2026-06-05
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   open    100 non-null    float64
 1   high    100 non-null    float64
 2   low     100 non-null    float64
 3   close   100 non-null    float64
 4   volume  100 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 4.7 KB
None
Inspecting the numerical summmary of the data :
             open        high         low       close        volume
count  100.000000  100.000000  100.000000  100.000000  1.000000e+02
mean   461.536000  467.315923  456.578661  462.006400  3.096220e+07
std     59.383225   59.593444   59.976022   59.999226  2.113764e+07
min    357.150000  364.230000  349.200000  352.830000  1.288957e+07
25%    393.340000  398.817500  388.610000  392.760000  1.970565e+07
50%    489.345000  495.580000  486.615000  492.070000  2.721609e+07
75% 

In [5]:
# 2. INspecting the data quality , looking for  missing data or incorrect datatypes
print("-"*50)
print("Sumamry of data structure : ")
print(df.info())
print("Summary of the null values in DF : ")
print(df.isnull().sum())



--------------------------------------------------
Sumamry of data structure : 
<class 'pandas.DataFrame'>
DatetimeIndex: 100 entries, 2026-10-27 to 2026-06-05
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   open    100 non-null    float64
 1   high    100 non-null    float64
 2   low     100 non-null    float64
 3   close   100 non-null    float64
 4   volume  100 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 4.7 KB
None
Summary of the null values in DF : 
open      0
high      0
low       0
close     0
volume    0
dtype: int64


In [6]:
# 3. inspecting the distribution in the data 
fig1=px.histogram(
    df,
    x="volume",
)
fig1.show()

fig2=px.histogram(
    df,
    x="open"
)
fig2.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# 4. inspecting the outliers

fig3=px.box(
    df,
    y="volume"
)
fig3.show()
fig4=px.box(
    df,
    y="open"
)
fig4.show()

In [ ]:
# 5. inspecting the coorelation bitween the variables in df
corr=df.corr(numeric_only=True)
fig5=px.imshow(
    corr,
    text_auto=True,
    title="Corelation heat map"
)
fig5.show()

In [ ]:
# streamlit app
st.title("EDA Snak App")
st.markdown("Automated Exploratory Data Analysis of Real Time Stock Data.")

# showcasignt the clean data
st.subheader("Clean Data (Last five days)")
st.dataframe(df.tail())

# distrubution analysis
st.subheader("Distribution Analysis")
st.plotly_chart(fig1)
st.plotly_chart(fig2)
#outliers analysis
st.subheader("Outliers analysis")
st.plotly_chart(fig3)
st.plotly_chart(fig4)
# Corelartion heat map
st.subheader("🌡️ Correlation Analysis")
st.plotly_chart(fig5) # Heatmap